# Apartado 1 | Naive-Bayes propio
Tabla con los resultados de la ejecución para los conjuntos de datos analizados (fuga_telefonia y wdbc).

In [ ]:
import pandas as pd
import numpy as np
from preprocesamiento import Dataset
from particionado import ValidacionSimple, ValidacionCruzada
from clasificador import ClasificadorNaiveBayes

def evaluar_naive_bayes(ruta_dataset, atributos_discretos, nombre_dataset):
    dataset = Dataset(ruta_dataset)
    datos = dataset.obtener_datos()
    
    # Instanciamos las dos estrategias de particionado requeridas
    estrategias = {
        "Validación Simple": ValidacionSimple(proporcion_test=0.3, numero_ejecuciones=10), # 10 ejecuciones para tener una desviación típica real
        "Validación Cruzada": ValidacionCruzada(numero_particiones=5)
    }
    
    resultados = []
    
    for nombre_est, validador in estrategias.items():
        validador.creaParticiones(len(datos))
        
        # Ejecutamos con y sin corrección de Laplace
        for laplace in [True, False]:
            clasificador = ClasificadorNaiveBayes(laplace=laplace)
            errores = []
            
            for particion in validador.particiones:
                datos_train = datos[particion["train"]]
                datos_test = datos[particion["test"]]
                
                # Entrenamiento y Clasificación
                clasificador.entrenamiento(datos_train, atributos_discretos, dataset.diccionarios)
                predicciones = clasificador.clasifica(datos_test, atributos_discretos, dataset.diccionarios)
                
                # Cálculo de error
                error = clasificador.error(datos_test, predicciones)
                errores.append(error)
                
            # Guardamos las métricas requeridas: promedio y desviación típica
            resultados.append({
                "Dataset": nombre_dataset,
                "Particionado": nombre_est,
                "Laplace": "Sí" if laplace else "No",
                "Error Medio (%)": np.mean(errores) * 100,
                "Desviación Típica": np.std(errores) * 100
            })
            
    return resultados

In [ ]:
# 1. Configuración WDBC: 30 atributos, todos continuos (False)
discretos_wdbc = [False] * 30
resultados_wdbc = evaluar_naive_bayes("wdbc.csv", discretos_wdbc, "WDBC")

# 2. Configuración Fuga Telefonía: Asumimos los 10 primeros continuos (False) y los 2 categóricos discretos (True)
discretos_fuga = [False] * 10 + [True, True]
resultados_fuga = evaluar_naive_bayes("fuga_telefonia.csv", discretos_fuga, "Fuga Telefonía")

# Combinamos y mostramos la tabla final usando Pandas
df_resultados = pd.DataFrame(resultados_wdbc + resultados_fuga)
df_resultados = df_resultados.round(4) # Redondeamos a 4 decimales para mayor limpieza
display(df_resultados)

**Breve análisis de los resultados:**
(Escribe aquí vuestras observaciones. Comenta en qué dataset y estrategia se obtiene menor error. Para discutir el efecto Laplace, fíjate en la tabla generada: ¿cambian los errores entre la fila donde "Laplace" es Sí y donde es No? Si en `fuga_telefonia` hay alguna mejora, es porque Laplace ha evitado ceros en las columnas categóricas de competidores. En `wdbc`, como no hay variables discretas, Laplace no debería tener ningún efecto porque la distribución normal no lo usa).